# Embedding Fine-Tuning — Colab GPU Notebook
Run this notebook on a **Colab T4 GPU** (Runtime > Change runtime type > T4 GPU).

It clones your repo, installs deps, downloads PubMedQA, builds chunk versions,
fine-tunes the embedding model via contrastive learning, and saves the result
so you can pull it back down locally for the API/dashboard.

In [ ]:
# 1. Clone your repo (push this project to GitHub first, then clone it here)
!git clone https://github.com/ananyanair12/domain-rag-pubmedqa.git
%cd domain-rag-pubmedqa

In [ ]:
# 2. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 3. Set environment variables (use Colab secrets or paste directly for a quick run)
import os
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"
os.environ["BASELINE_EMBEDDING_MODEL"] = "sentence-transformers/all-MiniLM-L6-v2"
os.environ["FINETUNED_EMBEDDING_PATH"] = "./models/finetuned-embedder-pubmedqa"
os.environ["SELECTED_CHUNK_SIZE"] = "512" 

In [ ]:
# 4. Download and split PubMedQA into train / held-out eval pools
!python data/download_pubmedqa.py

In [ ]:
# 5. Build chunked corpus versions (256 / 512 / 1024) for the ablation
!python src/chunking.py

In [ ]:
# 6. Fine-tune the embedding model (GPU-heavy step — this is why we're on Colab)
!python src/finetune_embeddings.py

In [ ]:
# 7. Build both baseline and fine-tuned vector collections at the selected chunk size
!python src/vector_store.py

In [ ]:
# 8. Run the core thesis result: precision@5 before vs. after fine-tuning
!python src/evaluate_retrieval.py

In [ ]:
# 9. (Optional, slower) Run the full chunk-size ablation across 256/512/1024
from src.evaluate_retrieval import run_chunk_size_ablation
run_chunk_size_ablation(embedder_kind="finetuned")

In [ ]:
# 10. (Optional) Run RAGAs evaluation — needs GROQ_API_KEY set above
!python src/evaluate_ragas.py

## 11. Download your results
Zip the fine-tuned model, chroma store, and results folder so you can pull
them down to your local machine and run the API + dashboard from there.

In [ ]:
!zip -r artifacts.zip models/ chroma_store/ results/
from google.colab import files
files.download("artifacts.zip")